In [ ]:
from google.colab import drive
import os
import pandas as pd
from scipy import stats
import numpy as np

In [ ]:
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/data'

# 3. Automatically change the working directory so relative paths work
if os.path.exists(DATA_DIR):
    os.chdir(DATA_DIR)
    print(f"Successfully connected! Current working directory: {os.getcwd()}")
else:
    print(f"Error: The folder '{DATA_DIR}' could not be found.")
    print("Make sure to add a shortcut of the 'data' folder to your 'My Drive'.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully connected! Current working directory: /content/drive/.shortcut-targets-by-id/1_ib6SYLLQ6Asjybns0GSlgBDwFQrnmkj/data


In [ ]:
# Load all datasets
chlorophylla1 = pd.read_csv("cleaned/ChlorophyllaConcentrationSarasota_cleaned.csv")

chlorophylla1

In [ ]:
chlorophylla1.head(25)

,time,latitude,longitude,chlor_a
0,2020-06-16T12:00:00Z,27.99375,276.50625,0.163719
1,2020-06-16T12:00:00Z,27.99375,276.54375,0.162463
2,2020-06-16T12:00:00Z,27.99375,276.61875,0.199357
3,2020-06-16T12:00:00Z,27.99375,276.65625,0.206935
4,2020-06-16T12:00:00Z,27.99375,276.69375,0.246541
5,2020-06-16T12:00:00Z,27.99375,276.73125,0.299513
6,2020-06-16T12:00:00Z,27.99375,276.76875,0.351332
7,2020-06-16T12:00:00Z,27.99375,276.80625,0.440458
8,2020-06-16T12:00:00Z,27.99375,276.88125,0.500225
9,2020-06-16T12:00:00Z,27.99375,276.91875,0.801173


In [ ]:
# Convert the time column into a datetime object
# so we can extract information such as year and month
chlorophylla1["time"] = pd.to_datetime(chlorophylla1["time"])

# Extract the year to capture long-term changes
chlorophylla1["year"] = chlorophylla1["time"].dt.year

# Extract the month to capture seasonal patterns
chlorophylla1["month"] = chlorophylla1["time"].dt.month

# Extract the day of the month
chlorophylla1["day"] = chlorophylla1["time"].dt.day

# Extract the day of the year to represent the position
# within the annual cycle
chlorophylla1["day_of_year"] = chlorophylla1["time"].dt.dayofyear

# Extract the hour to capture possible time-of-day patterns
chlorophylla1["hour"] = chlorophylla1["time"].dt.hour

In [ ]:
# Latitude of Mote Marine Laboratory
mote_lat = 27.333

# Longitude of Mote Marine Laboratory
# Converted from -82.577 to the 0–360 longitude format
# used by the dataset
mote_lon = 277.423

# Convert observation latitude to radians
lat1 = np.radians(chlorophylla1["latitude"])

# Convert observation longitude to radians
lon1 = np.radians(chlorophylla1["longitude"])

# Convert Mote's coordinates to radians
lat2 = np.radians(mote_lat)
lon2 = np.radians(mote_lon)

# Calculate the differences in latitude and longitude
dlat = lat1 - lat2
dlon = lon1 - lon2

# Haversine formula for calculating distance
# between two points on Earth's surface
a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2)
    * np.sin(dlon / 2) ** 2
)

# Calculate the angular distance
c = 2 * np.arcsin(np.sqrt(a))

# Earth's radius in kilometers
earth_radius = 6371

# Calculate distance from each observation to Mote
chlorophylla1["distance_from_mote_km"] = (
    earth_radius * c
)

# Calculate approximate north/south distance in kilometers
# Positive = north of Mote
# Negative = south of Mote
chlorophylla1["north_south_distance_km"] = (
    (chlorophylla1["latitude"] - mote_lat) * 111.32
)

# Calculate approximate east/west distance in kilometers
# Positive = east of Mote
# Negative = west of Mote
chlorophylla1["east_west_distance_km"] = (
    (chlorophylla1["longitude"] - mote_lon)
    * 111.32
    * np.cos(np.radians(chlorophylla1["latitude"]))
)

# Difference in latitude relative to Mote
chlorophylla1["latitude_difference"] = (
    chlorophylla1["latitude"] - mote_lat
)

# Difference in longitude relative to Mote
chlorophylla1["longitude_difference"] = (
    chlorophylla1["longitude"] - mote_lon
)

In [ ]:
# Sine representation of the month to capture seasonal cycles
chlorophylla1["month_sin"] = np.sin(
    2 * np.pi * chlorophylla1["month"] / 12
)

# Cosine representation of the month to capture seasonal cycles
chlorophylla1["month_cos"] = np.cos(
    2 * np.pi * chlorophylla1["month"] / 12
)

In [ ]:
# Chlorophyll × distance
chlorophylla1["chlor_a_x_distance"] = chlorophylla1["chlor_a"] * chlorophylla1["distance_from_mote_km"]

# Chlorophyll × north/south position
chlorophylla1["chlor_a_x_north_south"] = chlorophylla1["chlor_a"] * chlorophylla1["north_south_distance_km"]

# Chlorophyll × east/west position
chlorophylla1["chlor_a_x_east_west"] = chlorophylla1["chlor_a"] * chlorophylla1["east_west_distance_km"]

# Chlorophyll × month
chlorophylla1["chlor_a_x_month_sin"] = chlorophylla1["chlor_a"] * chlorophylla1["month_sin"]
chlorophylla1["chlor_a_x_month_cos"] = chlorophylla1["chlor_a"] * chlorophylla1["month_cos"]

# Chlorophyll × time of day
chlorophylla1["chlor_a_x_hour"] = chlorophylla1["chlor_a"] * chlorophylla1["hour"]

In [ ]:
chlorophylla1.head(25)

,time,latitude,longitude,chlor_a,year,month,day,day_of_year,hour,distance_from_mote_km,...,latitude_difference,longitude_difference,month_sin,month_cos,chlor_a_x_distance,chlor_a_x_north_south,chlor_a_x_east_west,chlor_a_x_month_sin,chlor_a_x_month_cos,chlor_a_x_hour
0,2020-06-16 12:00:00+00:00,27.99375,276.50625,0.163719,2020,6,16,168,12,116.401923,...,0.66075,-0.91675,1.224647e-16,-1.0,19.057211,12.042303,-14.753104,2.004980e-17,-0.163719,1.964628
1,2020-06-16 12:00:00+00:00,27.99375,276.54375,0.162463,2020,6,16,168,12,113.561386,...,0.66075,-0.87925,1.224647e-16,-1.0,18.449548,11.949932,-14.041087,1.989601e-17,-0.162463,1.949559
2,2020-06-16 12:00:00+00:00,27.99375,276.61875,0.199357,2020,6,16,168,12,108.035100,...,0.66075,-0.80425,1.224647e-16,-1.0,21.537513,14.663615,-15.759958,2.441415e-17,-0.199357,2.392280
3,2020-06-16 12:00:00+00:00,27.99375,276.65625,0.206935,2020,6,16,168,12,105.357471,...,0.66075,-0.76675,1.224647e-16,-1.0,21.802174,15.221057,-15.596299,2.534226e-17,-0.206935,2.483223
4,2020-06-16 12:00:00+00:00,27.99375,276.69375,0.246541,2020,6,16,168,12,102.742826,...,0.66075,-0.72925,1.224647e-16,-1.0,25.330335,18.134259,-17.672549,3.019258e-17,-0.246541,2.958494
5,2020-06-16 12:00:00+00:00,27.99375,276.73125,0.299513,2020,6,16,168,12,100.196096,...,0.66075,-0.69175,1.224647e-16,-1.0,30.010010,22.030569,-20.365629,3.667974e-17,-0.299513,3.594153
6,2020-06-16 12:00:00+00:00,27.99375,276.76875,0.351332,2020,6,16,168,12,97.722593,...,0.66075,-0.65425,1.224647e-16,-1.0,34.333087,25.842126,-22.594093,4.302578e-17,-0.351332,4.215986
7,2020-06-16 12:00:00+00:00,27.99375,276.80625,0.440458,2020,6,16,168,12,95.328015,...,0.66075,-0.61675,1.224647e-16,-1.0,41.988025,32.397781,-26.702220,5.394060e-17,-0.440458,5.285501
8,2020-06-16 12:00:00+00:00,27.99375,276.88125,0.500225,2020,6,16,168,12,90.800413,...,0.66075,-0.54175,1.224647e-16,-1.0,45.420594,36.793860,-26.637730,6.125984e-17,-0.500225,6.002694
9,2020-06-16 12:00:00+00:00,27.99375,276.91875,0.801173,2020,6,16,168,12,88.680744,...,0.66075,-0.50425,1.224647e-16,-1.0,71.048594,58.930012,-39.710502,9.811536e-17,-0.801173,9.614073


In [ ]:
os.chdir(DATA_DIR+"/engineered")
chlorophylla1.to_csv("ChlorophyllaConcentrationSarasota_engineered.csv", index=False)
print("Saved cleaned dataset to ChlorophyllaConcentrationSarasota_engineered.csv")

Saved cleaned dataset to ChlorophyllaConcentrationSarasota_engineered.csv


chirps20GlobalDailyP05_0878_d358_0bc6.txt

This is a spatial grid (31 latitudes x 35 longitudes) and repeats daily. Latitude and longitude are not constant here.

Removing the units row and converting to numeric like we did with the other files.

In [ ]:
chirps = chirps.iloc[1:].copy()

for col in ["latitude", "longitude", "precip"]:
    chirps[col] = pd.to_numeric(chirps[col], errors="coerce")

precip is missing about 3/4 of the time. Precipitation is only estimated over land so the grid cells that are ocean have missing values. Data is missing since June 2024. There are also gaps in the data from 2017-2019 and 2021-2023.

In [ ]:
ocean_cells = chirps.groupby(["latitude", "longitude"])["precip"].apply(lambda s: s.isna().all())
print("Ocean (always-NaN) grid cells:", ocean_cells.sum(), "/", len(ocean_cells))

fully_missing_days = chirps.groupby("time")["precip"].apply(lambda s: s.isna().all())
print("Fully-missing days:", fully_missing_days.sum(), "/", fully_missing_days.shape[0])

Ocean (always-NaN) grid cells: 591 / 1085
Fully-missing days: 739 / 1644


Turning the ocean-mask pattern into an explicit is_land feature. A cell that never has a value anywhere in the dataset is ocean. A cell that has a value on at least one date is land.

In [ ]:
cell_has_any_value = chirps.groupby(["latitude", "longitude"])["precip"].transform(lambda s: s.notna().any())
chirps["is_land"] = cell_has_any_value

Standardizing precip, like we did with the other files.

In [ ]:
chirps_scaler = StandardScaler()
chirps["precip"] = chirps_scaler.fit_transform(chirps[["precip"]])

Cyclically encoding month for seasonality purposes, like we did with the other files.

In [ ]:
chirps["time"] = pd.to_datetime(chirps["time"], utc=True, errors="coerce")
chirps["month"] = chirps["time"].dt.month
chirps["month_sin"] = np.sin(2 * np.pi * chirps["month"] / 12)
chirps["month_cos"] = np.cos(2 * np.pi * chirps["month"] / 12)